<a href="https://colab.research.google.com/github/vpattangi/emoji-reddit-data-cleaning/blob/main/emoji_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Emoji Use In Online Discourse

This project investigates how emojis, specifically the skull (💀) and loudly crying face (😭), function as linguistic markers of emotional intensity in digital communication.

Using a large-scale dataset of Reddit comments (May 2019), this study analyzes over one million entries to explore how emoji usage varies across different online communities, including sports, gaming, memes, and political discussions.

The project focuses on identifying patterns such as:
- Repeated emoji usage as a form of emotional amplification
- Differences between informal (emoji-rich) and formal (emoji-free) communication
- The relationship between topic context and expressive language

By combining quantitative data analysis with linguistic theory, this study highlights how modern online discourse blends text and visual symbols to convey tone, humor, and exaggerated emotion.

## Cleaning The Dataset

####Load Dataset

In [33]:
import pandas as pd

df = pd.read_csv("/content/kaggle_RC_2019-05.csv")
df.head()

,subreddit,body,controversiality,score
0,gameofthrones,Your submission has been automatically removed...,0,1
1,aww,"Dont squeeze her with you massive hand, you me...",0,19
2,gaming,It's pretty well known and it was a paid produ...,0,3
3,news,You know we have laws against that currently c...,0,10
4,politics,"Yes, there is a difference between gentle supp...",0,1


####Filter for emojis

In [34]:
filtered = df[df["body"].str.contains("💀|😭", na=False)]
filtered.head()

,subreddit,body,controversiality,score
5123,AskReddit,"Beading. \n\nWatching my pet chickens, when I ...",0,1
5232,FortNiteBR,Lmao 💀💀 y’all some funny bulls,0,2
5248,funny,"So technically apps that have ""lite"" ar the en...",0,3
6918,RoastMe,🙌🖕💀 go to hell with ur kleenexs too,0,-1
8806,gameofthrones,Skyrim is probably the closest we’re gonna get...,0,2


####Save Cleaned Dataset

In [35]:
filtered.to_csv("cleaned_emoji_data.csv", index=False)

####Download Cleaned Dataset

In [36]:
files.download("cleaned_emoji_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Reddit Category Classification

To better understand how language and emoji usage vary across different types of online communities, each Reddit comment is categorized based on its subreddit.

Subreddits are grouped into broader thematic categories, including:
- Sports (e.g., NBA, NFL)
- Gaming (e.g., Fortnite, Minecraft)
- Memes/Humor
- Politics/News
- Other

This classification allows the dataset to be analyzed at a higher level, making it possible to compare how different communities communicate.

For example, meme and gaming communities are expected to use more emojis and exaggerated expressions, while political and news-related discussions tend to be more formal and emoji-free.

By organizing the data into categories, this step enables both quantitative analysis (e.g., frequency of emoji use) and qualitative insights into how tone and emotional expression differ across contexts.

####Load Cleaned Dataset

In [37]:
import pandas as pd

df = pd.read_csv("cleaned_emoji_data.csv")
df.head()

,subreddit,body,controversiality,score
0,AskReddit,"Beading. \n\nWatching my pet chickens, when I ...",0,1
1,FortNiteBR,Lmao 💀💀 y’all some funny bulls,0,2
2,funny,"So technically apps that have ""lite"" ar the en...",0,3
3,RoastMe,🙌🖕💀 go to hell with ur kleenexs too,0,-1
4,gameofthrones,Skyrim is probably the closest we’re gonna get...,0,2


#### Creating Subreddit Categories

In [38]:
def categorize_topic(subreddit):
    subreddit = str(subreddit).lower()

    # sports
    if any(word in subreddit for word in ["nba", "nfl", "soccer", "sports", "baseball"]):
        return "sports"

    # gaming
    elif any(word in subreddit for word in ["gaming", "fortnite", "apex", "cod", "minecraft"]):
        return "gaming"

    # memes / humor
    elif any(word in subreddit for word in ["funny", "memes", "dankmemes"]):
        return "memes"

    # politics / serious
    elif any(word in subreddit for word in ["politics", "news", "worldnews"]):
        return "politics"

    else:
        return "other"

#### Apply Category Sorting to Dataset

In [39]:
df["topic"] = df["subreddit"].apply(categorize_topic)

df.head()

,subreddit,body,controversiality,score,topic
0,AskReddit,"Beading. \n\nWatching my pet chickens, when I ...",0,1,other
1,FortNiteBR,Lmao 💀💀 y’all some funny bulls,0,2,gaming
2,funny,"So technically apps that have ""lite"" ar the en...",0,3,memes
3,RoastMe,🙌🖕💀 go to hell with ur kleenexs too,0,-1,other
4,gameofthrones,Skyrim is probably the closest we’re gonna get...,0,2,other


#### Lines Per Category Count

In [40]:
df["topic"].value_counts()

,count
topic,
other,388
gaming,52
memes,42
sports,18
politics,4


#### Compare Emoji Usage by Topic

In [41]:
def get_emoji_type(text):
    if pd.isna(text):
        return "none"
    has_skull = "💀" in text
    has_cry = "😭" in text

    if has_skull and has_cry:
        return "mixed"
    elif has_skull:
        return "skull"
    elif has_cry:
        return "crying"
    else:
        return "none"

df["emoji_type"] = df["body"].apply(get_emoji_type)

In [42]:
pd.crosstab(df["topic"], df["emoji_type"])

emoji_type,crying,mixed,skull
topic,,,
gaming,45,1,6
memes,38,0,4
other,352,0,36
politics,4,0,0
sports,12,0,6


#### Examples Per Topic

In [43]:
df[df["topic"] == "sports"].head(5)
df[df["topic"] == "gaming"].head(5)
df[df["topic"] == "memes"].head(5)

,subreddit,body,controversiality,score,topic,emoji_type
2,funny,"So technically apps that have ""lite"" ar the en...",0,3,memes,crying
6,funny,That's an ugly fucking kid tho I'm dead 😭,0,1,memes,crying
10,memes,If only this were true and we could actually m...,0,3,memes,crying
30,memes,RIP unsettled Tom he will always be remembered 😭😭,0,7,memes,crying
34,funny,"&gt;...but how could a shoe, talk to you, the ...",0,2,memes,crying


#### Save Final Dataset

In [44]:
df.to_csv("final_emoji_topic_dataset.csv", index=False)

##Emoji Intensification Classification

Emoji intensification is defined as the repetition of emojis within a single comment to amplify emotional meaning.

This functions as a paralinguistic feature of digital language, where repeated symbols (e.g., 💀💀💀, 😭😭😭) enhance the perceived intensity of an expression beyond what text alone conveys.

In this analysis, comments are categorized into:
- Single emoji use (casual expression)
- Repeated emoji use (intensified emotion)

This distinction allows for a more nuanced understanding of how emotional emphasis is constructed in online communication.

#### Emoji Intensification Identification

In [45]:
import pandas as pd
import re

# load dataset
df = pd.read_csv("cleaned_emoji_data.csv")

# detect repeated emojis
def has_repeated_emoji(text):
    if pd.isna(text):
        return False
    return bool(re.search(r'(💀){2,}|(😭){2,}', text))

# filter
intensified_comments = df[df["body"].apply(has_repeated_emoji)]

# print results
print("Emoji Intensification Comments:")
for i, row in intensified_comments.iterrows():
    print(f"Subreddit: {row['subreddit']}")
    print(f"Comment: {row['body']}")
    print("-" * 50)

# save to CSV
intensified_comments.to_csv("emoji_intensification.csv", index=False)

print("\nSaved to emoji_intensification.csv")

Emoji Intensification Comments:
Subreddit: FortNiteBR
Comment: Lmao 💀💀 y’all some funny bulls
--------------------------------------------------
Subreddit: gameofthrones
Comment: Skyrim is probably the closest we’re gonna get for a while 😭😭
--------------------------------------------------
Subreddit: FortNiteBR
Comment: I WANT DREAM FEET 😭😭
--------------------------------------------------
Subreddit: freefolk
Comment: Yeah. I never liked Theon. Never gave a shit. Then just watching him kill like 25 dudes alone. The heart it takes to be the last man standing in the position and he has 6 friggin fingers. Then crows nest over there tipped me over with the “...you’re a good man.” Ugh🥺🥺😭😭
--------------------------------------------------
Subreddit: nba
Comment: Imagine being a bucks fan and really thinking Giannis is the best player in the world 💀💀💀💀💀
--------------------------------------------------
Subreddit: memes
Comment: If only this were true and we could actually make stan lee 😭😭

## Serious-Tone Reddit Comment Classification

Serious tone is defined as the absence of emojis in comments, particularly within politics and news-related subreddits.

This form of communication relies on textual language alone and reflects a more formal register, where emotional expression is minimized. In contrast to emoji-based discourse, serious comments prioritize clarity, argumentation, and informational content.

This category serves as a baseline for comparison, allowing the analysis to examine how the presence or absence of emojis influences tone, emotional intensity, and communicative style.

#### Reddit Classification for Serious-Tone Comments

In [46]:
import pandas as pd
import re

# load original kaggle dataset
df = pd.read_csv("kaggle_RC_2019-05.csv")


# filter politics/news
def is_politics_news(subreddit):
    subreddit = str(subreddit).lower()
    return any(word in subreddit for word in ["politics", "news", "worldnews"])


# check no emojis
def has_no_emoji(text):
    if pd.isna(text):
        return False
    return not bool(re.search(r'💀|😭', text))

# filter dataset
serious_comments = df[
    df["subreddit"].apply(is_politics_news) &
    df["body"].apply(has_no_emoji)
]

# print sample
print("Sample of Serious Tone:")

for i, row in serious_comments.head(20).iterrows():  # only first 20
    print(f"Subreddit: {row['subreddit']}")
    print(f"Comment: {row['body']}")
    print("-" * 50)

# save to CSV
serious_comments.to_csv("politics_news_no_emoji.csv", index=False)

print("\nSaved to politics_news_no_emoji.csv")
print("Total rows:", len(serious_comments))

Sample of Serious Tone:
Subreddit: news
Comment: You know we have laws against that currently correct? Or are you just willfully ignorant of gun laws in the US.
--------------------------------------------------
Subreddit: politics
Comment: Yes, there is a difference between gentle suppression and hard suppression.  Neither are good things.
--------------------------------------------------
Subreddit: worldnews
Comment: How many millions have to suffer and die for them to feel any of that pain though?
--------------------------------------------------
Subreddit: news
Comment: No, not really at all. It happens too often but it's ridiculous to pretend that applies to everyone.
--------------------------------------------------
Subreddit: news
Comment: This is disgusting every week
--------------------------------------------------
Subreddit: worldnews
Comment: Trump's administration is lying on Fox news? I can't believe it!
--------------------------------------------------
Subreddit: wo

## Full Uploaded And Organized Dataset + Dashboard

The final dataset integrates multiple processed data sources into a single, structured file for analysis.

It combines:
- Emoji-containing comments (💀, 😭) from the cleaned dataset
- Emoji-free comments from politics and news subreddits in the original dataset

Each comment is enriched with additional features, including:
- Subreddit (source community)
- Category (e.g., sports, gaming, memes, politics)
- Emoji type (skull, crying, mixed, or none)
- Emoji intensity (single or repeated)
- Tone classification (casual, intensified emotion, or serious)

This structured dataset enables direct comparison between different communication styles, allowing the analysis to examine how emoji use, emotional intensity, and topic context interact in online discourse.

By organizing the data in this way, the project moves beyond simple filtering and creates a comprehensive framework for both quantitative and qualitative linguistic analysis.

Dashboard:

This efficiently classifies and allows to check emoji specific data, as well as specific Reddit emoji data being used for this project.

#### Intall Necessary Packages

In [ ]:
!pip install ipywidgets

#### Data Preperation

In [47]:
import pandas as pd

# load original dataset
df = pd.read_csv("kaggle_RC_2019-05.csv")

# emoji type
def detect_emoji_type(text):
    text = str(text)
    if "😭" in text:
        return "crying"
    elif "💀" in text:
        return "skull"
    else:
        return "none"

df["emoji_type"] = df["body"].apply(detect_emoji_type)


# emoji intensity
def emoji_intensity(text):
    text = str(text)
    return text.count("😭") + text.count("💀")

df["emoji_intensity"] = df["body"].apply(emoji_intensity)


# category
def categorize_subreddit(sub):
    sub = str(sub).lower()

    if any(x in sub for x in ["nba", "nfl", "soccer", "hockey"]):
        return "sports"
    elif any(x in sub for x in ["gaming", "xbox", "pcgaming"]):
        return "gaming"
    elif any(x in sub for x in ["memes", "funny"]):
        return "memes"
    elif any(x in sub for x in ["politics", "news"]):
        return "politics"
    else:
        return "other"

df["category"] = df["subreddit"].apply(categorize_subreddit)


# tone
def classify_tone(row):
    if row["category"] == "politics" and row["emoji_type"] == "none":
        return "serious"
    else:
        return "informal"

df["tone"] = df.apply(classify_tone, axis=1)


# final dataset
final_df = df.copy()

# save it
final_df.to_csv("FINAL_FULL_DATASET.csv", index=False)

print("FINAL DATASET CREATED")

FINAL DATASET CREATED


In [48]:
import pandas as pd

df = pd.read_csv("FINAL_FULL_DATASET.csv")
print(df.columns)

files.download("FINAL_FULL_DATASET.csv")

Index(['subreddit', 'body', 'controversiality', 'score', 'emoji_type',
       'emoji_intensity', 'category', 'tone'],
      dtype='object')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#### Dashboard Interactive Button System

In [51]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

df = pd.read_csv("FINAL_FULL_DATASET.csv")

print("Columns:", df.columns)
print("Total rows:", len(df))

output = widgets.Output()

def category_data(emoji):
    sports_terms = ["nba", "nfl", "soccer", "basketball", "football", "hockey"]
    gaming_terms = ["gaming", "ps5", "xbox", "pcgaming"]
    meme_terms = ["meme", "memes", "funny", "dankmemes"]

    def match_category(sub):
        sub = str(sub).lower()
        return (
            any(x in sub for x in sports_terms) or
            any(x in sub for x in gaming_terms) or
            any(x in sub for x in meme_terms)
        )

    return df[
        (df["emoji_type"] == emoji) &
        (df["subreddit"].apply(match_category))
    ][["subreddit", "body", "category", "emoji_intensity"]]

def full_data(emoji):
    return df[df["emoji_type"] == emoji][
        ["subreddit", "body", "category", "emoji_intensity"]
    ]

def serious_full():
    return df[df["emoji_type"] == "none"][
        ["subreddit", "body", "category"]
    ]

def serious_news_politics():
    terms = ["news", "politics"]
    return df[
        (df["emoji_type"] == "none") &
        (df["subreddit"].str.lower().apply(lambda x: any(t in x for t in terms)))
    ][["subreddit", "body", "category"]]

def show_table(data):
    with output:
        clear_output()
        print(f"Total rows: {len(data)}")
        display(data)

def emoji_menu(emoji_name):
    btn_full = widgets.Button(description="📊 Full Data")
    btn_cat = widgets.Button(description="🎯 Sports/Gaming/Memes")

    btn_full.on_click(lambda x: show_table(full_data(emoji_name)))
    btn_cat.on_click(lambda x: show_table(category_data(emoji_name)))

    return widgets.VBox([btn_full, btn_cat])

def serious_menu():
    btn_full = widgets.Button(description="📊 Full Data")
    btn_np = widgets.Button(description="📰 News/Politics")

    btn_full.on_click(lambda x: show_table(serious_full()))
    btn_np.on_click(lambda x: show_table(serious_news_politics()))

    return widgets.VBox([btn_full, btn_np])

btn_cry = widgets.Button(description="😭 Crying Emoji")
btn_skull = widgets.Button(description="💀 Skull Emoji")
btn_serious = widgets.Button(description="📰 Serious Tone")

def click_cry(b):
    with output:
        clear_output()
        display(emoji_menu("crying"))

def click_skull(b):
    with output:
        clear_output()
        display(emoji_menu("skull"))

def click_serious(b):
    with output:
        clear_output()
        display(serious_menu())

btn_cry.on_click(click_cry)
btn_skull.on_click(click_skull)
btn_serious.on_click(click_serious)

display(widgets.HBox([btn_cry, btn_skull, btn_serious]))
display(output)

Columns: Index(['subreddit', 'body', 'controversiality', 'score', 'emoji_type',
       'emoji_intensity', 'category', 'tone'],
      dtype='object')
Total rows: 1000000


Output()